# F-003-3c: YOLOv8n Grayscale (1ch) 入力対応 学習 Notebook

転倒検出用の人物検出モデルを Grayscale (1チャネル) 入力で学習し、
ONNX / TFLite INT8 形式にエクスポートする。

## 案A: Grayscaleで再学習
- カスタムモデル YAML で `ch: 1` を指定
- データセット画像をGrayscale変換してから学習
- MCUカメラ (Grayscale) と同条件で学習するため最も精度が高い

## 前提条件
- Google Colab (GPU ランタイム: T4 推奨)
- Google Drive に `fall_detection_dataset.zip` をアップロード済み

## ワークフロー
1. GPU確認・環境構築
2. データセット準備 (Grayscale変換)
3. カスタムモデルYAML作成
4. モデル学習 (Grayscale入力)
5. 精度評価 (mAP) + RGB版との比較
6. ONNX エクスポート
7. TFLite FP32/INT8 変換
8. 入力形状・サイズ検証
9. 成果物ダウンロード

---
## Step 1: GPU確認・環境構築

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')

In [ ]:
# Ultralytics YOLOv8 と変換ツールのインストール
!pip install -q ultralytics onnx onnx2tf onnxsim
print('=== インストール完了 ===')

---
## Step 2: データセット準備 (Grayscale変換)

### 事前準備 (ローカルPCで実行)

```bash
cd mimamori-sense/dataset/merged
zip -r fall_detection_dataset.zip images/ labels/
```

作成した `fall_detection_dataset.zip` を Google Drive のマイドライブ直下にアップロードしてください。

### Grayscale変換について

UltralyticsのデータローダーはデフォルトでRGB画像を読み込みます。
カスタムモデルYAMLで `ch: 1` を指定した場合、モデルのアーキテクチャは
1ch入力になりますが、データローダー側の変換も必要です。

対応方法:
1. **データセット画像を事前にGrayscale変換** (このNotebookで実施)
2. UltralyticsのデータローダーでRGB画像をロードし、`ch: 1` の場合は
   モデルの最初のConv層がRGBチャネルを平均して1chとして処理する

方法1が確実なため、このNotebookでは事前変換を行います。

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

WORK_DIR = '/content/yolo_grayscale_train'
DATASET_DIR = os.path.join(WORK_DIR, 'dataset')
DATASET_GRAY_DIR = os.path.join(WORK_DIR, 'dataset_gray')
DATASET_ZIP = '/content/drive/MyDrive/fall_detection_dataset.zip'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'作業ディレクトリ: {WORK_DIR}')

# データセット展開
if not os.path.isdir(DATASET_DIR):
    if os.path.isfile(DATASET_ZIP):
        print('データセット展開中...')
        !mkdir -p {DATASET_DIR} && unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        print('展開完了')
    else:
        print(f'ERROR: {DATASET_ZIP} が見つかりません')
        print('Google Drive にアップロードしてください')
else:
    print('データセットは展開済みです')

# 検証
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        lbl_count = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
        print(f'  {split}: images={img_count}, labels={lbl_count}')
    else:
        print(f'  WARNING: {img_dir} が見つかりません')

In [ ]:
# データセット画像をGrayscale変換
# ラベルファイルはそのままコピー（バウンディングボックス座標は変わらない）

from PIL import Image
import shutil

print('=== Grayscale データセット作成 ===')

converted_count = 0
for split in ['train', 'val', 'test']:
    # images ディレクトリ
    src_img_dir = os.path.join(DATASET_DIR, 'images', split)
    dst_img_dir = os.path.join(DATASET_GRAY_DIR, 'images', split)
    os.makedirs(dst_img_dir, exist_ok=True)

    # labels ディレクトリ (そのままコピー)
    src_lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    dst_lbl_dir = os.path.join(DATASET_GRAY_DIR, 'labels', split)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    if not os.path.isdir(src_img_dir):
        print(f'  WARNING: {src_img_dir} が見つかりません')
        continue

    # 画像をGrayscale変換
    img_files = [f for f in os.listdir(src_img_dir)
                 if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    for img_file in img_files:
        src_path = os.path.join(src_img_dir, img_file)
        dst_path = os.path.join(dst_img_dir, img_file)
        try:
            img = Image.open(src_path).convert('L')  # Grayscale変換
            img.save(dst_path)
            converted_count += 1
        except Exception as e:
            print(f'  ERROR: {img_file}: {e}')

    # ラベルをコピー
    if os.path.isdir(src_lbl_dir):
        lbl_files = [f for f in os.listdir(src_lbl_dir) if f.endswith('.txt')]
        for lbl_file in lbl_files:
            shutil.copy2(
                os.path.join(src_lbl_dir, lbl_file),
                os.path.join(dst_lbl_dir, lbl_file)
            )

    img_count = len([f for f in os.listdir(dst_img_dir)
                     if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    lbl_count = len([f for f in os.listdir(dst_lbl_dir)
                     if f.endswith('.txt')])
    print(f'  {split}: images={img_count}, labels={lbl_count}')

print(f'\n合計 {converted_count} 枚の画像をGrayscaleに変換しました')

# 変換結果の確認（サンプル画像表示）
import matplotlib.pyplot as plt
sample_dir = os.path.join(DATASET_GRAY_DIR, 'images', 'train')
if os.path.isdir(sample_dir):
    samples = sorted(os.listdir(sample_dir))[:3]
    if samples:
        fig, axes = plt.subplots(1, len(samples), figsize=(12, 4))
        if len(samples) == 1:
            axes = [axes]
        for ax, s in zip(axes, samples):
            img = Image.open(os.path.join(sample_dir, s))
            ax.imshow(img, cmap='gray')
            ax.set_title(f'{s}\nmode={img.mode}, size={img.size}')
            ax.axis('off')
        plt.suptitle('Grayscale変換サンプル')
        plt.tight_layout()
        plt.show()

In [ ]:
# data.yaml 作成 (Grayscaleデータセット用)
data_yaml = f"""path: {DATASET_GRAY_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: person
"""

data_yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

print(f'data.yaml 作成完了: {data_yaml_path}')
print()
print(data_yaml)

---
## Step 3: カスタムモデルYAML作成

YOLOv8n 構成をベースに、入力チャネルのみ `ch: 1` に変更する。

### 学習方式の選択

| 方式 | 説明 | 精度期待 |
|------|------|----------|
| スクラッチ学習 | COCO事前学習重みなしでゼロから学習 | やや低い |
| 転移学習 (推奨) | YOLOv8n事前学習重みの最初のConv層のみ1ch用に調整 | 高い |

Ultralyticsは `ch` が事前学習重みと異なる場合、最初のConv層の重みを
自動的に調整します (3ch重みの平均を取って1chにする)。
このため転移学習が利用可能です。

ただし、**データセット画像を事前にGrayscale変換しておくことが重要**です。
Ultralyticsのデータローダーは `ch: 1` を指定してもデフォルトでRGB読み込みを
行うため、事前変換済みのGrayscale画像を使用することで確実に1ch学習できます。

In [ ]:
#############################################
# 学習設定
#############################################

# 入力設定
IMG_SIZE = 192
INPUT_CHANNELS = 1  # Grayscale

# 学習方式: 'transfer' (推奨) or 'scratch'
TRAIN_MODE = 'transfer'

# エポック数
EPOCHS = 200  # RGB版(100)より多めに設定。転移学習時は150でも可

# バッチサイズ
BATCH_SIZE = 64

print(f'学習設定:')
print(f'  入力: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS} (Grayscale)')
print(f'  学習方式: {TRAIN_MODE}')
print(f'  エポック数: {EPOCHS}')
print(f'  バッチサイズ: {BATCH_SIZE}')

In [ ]:
# カスタムモデル YAML を動的に生成
model_yaml_content = f"""# YOLOv8n-Grayscale: Fall detection model (Issue #103)
# Input: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS} (Grayscale)

nc: 1  # person only
ch: {INPUT_CHANNELS}  # Grayscale

scales:
  n: [0.33, 0.25, 1024]  # YOLOv8n standard

# YOLOv8 backbone
backbone:
  - [-1, 1, Conv, [64, 3, 2]]       # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]      # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]      # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]      # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]     # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]        # 9

# YOLOv8 head
head:
  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 6], 1, Concat, [1]]       # cat backbone P4
  - [-1, 3, C2f, [512]]             # 12

  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 4], 1, Concat, [1]]       # cat backbone P3
  - [-1, 3, C2f, [256]]             # 15 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]      # cat head P4
  - [-1, 3, C2f, [512]]             # 18 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]       # cat head P5
  - [-1, 3, C2f, [1024]]            # 21 (P5/32-large)

  - [[15, 18, 21], 1, Detect, [nc]]  # Detect(P3, P4, P5)
"""

model_yaml_path = os.path.join(WORK_DIR, 'yolov8n-grayscale-fall.yaml')
with open(model_yaml_path, 'w') as f:
    f.write(model_yaml_content)

print(f'モデルYAML作成完了: {model_yaml_path}')
print()
print(model_yaml_content)

In [ ]:
# モデル構造とパラメータ数を事前確認
from ultralytics import YOLO

# カスタムYAMLからモデル構築 (重みなし、アーキテクチャのみ)
model_check = YOLO(model_yaml_path)
print(f'\n=== YOLOv8n-Grayscale モデル情報 ===')
print(model_check.info())

# パラメータ数の確認
total_params = sum(p.numel() for p in model_check.model.parameters())
print(f'\n総パラメータ数: {total_params:,} ({total_params/1e6:.3f}M)')
print(f'推定INT8サイズ: {total_params/1024:.0f} KB')

# 参考値との比較
print(f'\n--- 参考 ---')
print(f'YOLOv8n (RGB, 3ch):  3.0M params, 3,149KB INT8')
print(f'YOLO-Fastest (顔認識): 0.24M params, 412KB INT8')
print(f'Arena上限:             432KB')
print(f'\n注意: YOLOv8n-Grayscale は最初のConv層のみ1/3になるため、')
print(f'パラメータ数はRGB版とほぼ同じ (~3.0M) です。')
print(f'サイズ削減が必要な場合は F-003-3b (pico/nano-slim) と組み合わせてください。')

---
## Step 4: モデル学習 (Grayscale入力)

### 転移学習モード (推奨)
- YOLOv8n COCO事前学習重み (`yolov8n.pt`) をロード
- `ch: 1` との不一致は Ultralytics が自動調整
  (最初のConv層の3ch重みを平均して1chに変換)
- 事前学習の特徴抽出能力を活用できるため、精度が高い

### スクラッチ学習モード
- ランダム初期化から学習
- エポック数を多く (200-300) 設定する必要がある

**注意:**
- Grayscale画像でも Ultralytics は内部でRGB(3ch)に変換して読み込む場合がある
- 事前にGrayscaleに変換した画像を使用し、`ch: 1` 指定で学習する
- 色相 (hsv_h) と彩度 (hsv_s) のaugmentationは無効化する

In [ ]:
from ultralytics import YOLO

if TRAIN_MODE == 'transfer':
    # 転移学習: COCO事前学習済み重みをロードし、ch=1に自動調整
    print('=== 転移学習モード ===')
    print('YOLOv8n COCO事前学習重みをロードし、最初のConv層を1ch用に調整します')
    model = YOLO(model_yaml_path).load('yolov8n.pt')
else:
    # スクラッチ学習
    print('=== スクラッチ学習モード ===')
    print('ランダム初期化から学習します。エポック数を多めに設定してください')
    model = YOLO(model_yaml_path)

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,           # GPU
    workers=2,
    project=WORK_DIR,
    name='train',
    exist_ok=True,
    # 学習パラメータ
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5.0,   # 転移学習でも安定性のためウォームアップを長めに
    # データ拡張 (Grayscale向け)
    hsv_h=0.0,          # 色相変換を無効化 (Grayscaleなので無意味)
    hsv_s=0.0,          # 彩度変換を無効化 (Grayscaleなので無意味)
    hsv_v=0.4,          # 明度変換のみ有効 (Grayscaleで重要)
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)

print('\n=== 学習完了 ===')

In [ ]:
# 学習曲線の表示
from IPython.display import Image, display
import os

results_png = os.path.join(WORK_DIR, 'train', 'results.png')
if os.path.exists(results_png):
    display(Image(filename=results_png, width=800))
else:
    print('学習結果の画像が見つかりません')

---
## Step 5: 精度評価 (mAP) + RGB版との比較

In [ ]:
# best.pt で検証データセットを評価
best_pt = os.path.join(WORK_DIR, 'train', 'weights', 'best.pt')
model = YOLO(best_pt)

metrics = model.val(
    data=data_yaml_path,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    split='val',
)

print(f'\n=== 検証データ評価結果 (Grayscale) ===')
print(f'mAP@0.5     : {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

# RGB版 (YOLOv8n, 100epochs) との比較
print(f'\n--- RGB版 YOLOv8n (100epochs) との比較 ---')
rgb_map50 = 67.8
rgb_recall = 59.3
gray_map50 = metrics.box.map50 * 100
gray_recall = metrics.box.mr * 100

map_diff = gray_map50 - rgb_map50
recall_diff = gray_recall - rgb_recall

print(f'mAP@0.5  : {gray_map50:.1f}% vs {rgb_map50:.1f}% (RGB) [{map_diff:+.1f}pt]')
print(f'Recall   : {gray_recall:.1f}% vs {rgb_recall:.1f}% (RGB) [{recall_diff:+.1f}pt]')

if abs(map_diff) <= 5:
    print('\n>>> 結果: Grayscale版はRGB版と同等の精度です (差分5pt以内)')
elif map_diff > 5:
    print('\n>>> 結果: Grayscale版はRGB版より精度が向上しました')
else:
    print(f'\n>>> 結果: Grayscale版はRGB版より精度が低下しています ({map_diff:+.1f}pt)')
    if map_diff > -10:
        print('    軽微な劣化のため許容範囲内です。メモリ効率向上のメリットが上回ります')
    else:
        print('    対策: エポック数を増やす、データ拡張を調整する、入力サイズを拡大する')

In [ ]:
# テストデータセットでも評価
test_metrics = model.val(
    data=data_yaml_path,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    split='test',
)

print(f'\n=== テストデータ評価結果 (Grayscale) ===')
print(f'mAP@0.5     : {test_metrics.box.map50:.4f} ({test_metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {test_metrics.box.map:.4f} ({test_metrics.box.map*100:.1f}%)')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')

---
## Step 6: ONNX エクスポート

In [ ]:
# ONNX エクスポート
model = YOLO(best_pt)

onnx_path = model.export(
    format='onnx',
    imgsz=IMG_SIZE,
    opset=11,
    simplify=True,
)

print(f'\nONNX エクスポート完了: {onnx_path}')
print(f'サイズ: {os.path.getsize(onnx_path)/1024:.1f} KB')

# ONNX 入力形状の確認
import onnx
onnx_model = onnx.load(onnx_path)
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'ONNX入力: {inp.name}, shape={shape}')
    if shape[-1] == 1 or shape[1] == 1:
        print('  -> 1ch (Grayscale) 入力を確認')
    else:
        print(f'  -> WARNING: {shape[-1]}ch 入力です。ch=1 が期待されます')

---
## Step 7: TFLite FP32/INT8 変換 (Grayscale対応)

In [ ]:
import numpy as np
import glob
import os
import onnx
from PIL import Image
import tensorflow as tf

FP32_PATH = os.path.join(WORK_DIR, 'model_grayscale_fp32.tflite')
INT8_PATH = os.path.join(WORK_DIR, 'model_grayscale_int8.tflite')

# --- Step 7a: ONNX入力形状の確認 ---
print('=== ONNX 入力形状確認 ===')
onnx_model = onnx.load(onnx_path)
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'ONNX入力: {inp.name}, shape={shape} (NCHW)')

onnx_shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
onnx_ch = onnx_shape[1]  # NCHW format
print(f'入力チャネル数: {onnx_ch}')

if onnx_ch == 1:
    print('-> ONNX は正しく 1ch です。onnx2tf の変換で問題が発生しています。')
elif onnx_ch == 3:
    print('-> ONNX が 3ch のままです。エクスポート時に ch=1 が反映されていません。')

# --- Step 7b: ONNX が3chの場合、1chに修正 ---
ONNX_1CH_PATH = os.path.join(WORK_DIR, 'model_grayscale_1ch.onnx')

if onnx_ch == 3:
    print('\n=== ONNX 入力を 3ch -> 1ch に修正 ===')
    import copy

    model_1ch = copy.deepcopy(onnx_model)

    # 入力テンソルの形状を変更: [1,3,192,192] -> [1,1,192,192]
    inp_tensor = model_1ch.graph.input[0]
    inp_tensor.type.tensor_type.shape.dim[1].dim_value = 1

    # 最初のConv層の重みを修正 (3ch -> 1ch: チャネル平均)
    first_conv_weight = None
    for init in model_1ch.graph.initializer:
        w = np.array(onnx.numpy_helper.to_array(init))
        if w.ndim == 4 and w.shape[1] == 3:  # [out_ch, in_ch=3, kH, kW]
            first_conv_weight = init
            print(f'最初のConv重み: {init.name}, shape={w.shape}')
            # 3ch重みを平均して1chに
            w_1ch = w.mean(axis=1, keepdims=True)  # [out_ch, 1, kH, kW]
            new_tensor = onnx.numpy_helper.from_array(w_1ch, name=init.name)
            init.CopyFrom(new_tensor)
            print(f'  -> 修正後: shape={w_1ch.shape}')
            break

    onnx.save(model_1ch, ONNX_1CH_PATH)
    print(f'1ch ONNX 保存: {ONNX_1CH_PATH}')
    onnx_path_for_convert = ONNX_1CH_PATH
else:
    onnx_path_for_convert = onnx_path

# --- Step 7c: onnx2tf で SavedModel 変換 ---
SAVED_MODEL_DIR = os.path.join(WORK_DIR, 'saved_model')
print(f'\n=== ONNX -> SavedModel (onnx2tf) ===')
os.system(f'onnx2tf -i {onnx_path_for_convert} -o {SAVED_MODEL_DIR} -osd 2>&1 | tail -15')

# SavedModel の入力形状を確認
if os.path.isdir(SAVED_MODEL_DIR):
    loaded = tf.saved_model.load(SAVED_MODEL_DIR)
    sig = loaded.signatures['serving_default']
    for name, spec in sig.structured_input_signature[1].items():
        print(f'SavedModel入力: {name}, shape={spec.shape}')

# --- Step 7d: FP32 TFLite 変換 ---
print('\n=== FP32 TFLite 変換 ===')
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
tflite_fp32 = converter.convert()
with open(FP32_PATH, 'wb') as f:
    f.write(tflite_fp32)

# FP32 入力形状の確認
interp = tf.lite.Interpreter(model_path=FP32_PATH)
interp.allocate_tensors()
inp_detail = interp.get_input_details()[0]
n, h, w, c = inp_detail['shape']
print(f'FP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')
print(f'入力形状: [{n}, {h}, {w}, {c}] (NHWC)')

if c != 1:
    print(f'ERROR: 入力チャネルが {c} です。1ch変換に失敗しています。')
    print('以降の処理をスキップします。')
else:
    # --- Step 7e: INT8 量子化 ---
    print('\n=== INT8 量子化 (Grayscale) ===')
    cal_dir = os.path.join(DATASET_GRAY_DIR, 'images', 'val')
    cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.jpg')))[:200]
    if not cal_images:
        cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.png')))[:200]
    print(f'キャリブレーション画像: {len(cal_images)}枚')

    def representative_dataset():
        for img_path in cal_images:
            img = Image.open(img_path).convert('L').resize((w, h))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.reshape(1, h, w, 1)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        print(f'INT8 TFLite: {os.path.getsize(INT8_PATH)/1024:.1f} KB')
        print('[PASS] INT8 量子化成功')
    except Exception as e:
        print(f'INT8 量子化エラー: {e}')

---
## Step 8: 入力形状・サイズ検証

### 受け入れ条件の確認

1. モデルの入力が `[1, 192, 192, 1]` であること
2. INT8量子化後も正常に動作すること
3. 入力バッファサイズが36,864バイト (192*192*1) であること

In [ ]:
import tensorflow as tf
import numpy as np

ARENA_LIMIT_KB = 432

print('=== 入力形状・サイズ検証 ===')
print()

# サイズ比較表
print('--- ファイルサイズ比較 ---')
for label, path in [('FP32', FP32_PATH), ('INT8', INT8_PATH)]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'{label}: {size_kb:.1f} KB ({size_kb/1024:.2f} MB)')

all_checks_passed = True

if os.path.exists(INT8_PATH):
    print(f'\n=== INT8 モデル詳細 ===')
    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    # 入力チェック
    inp_details = interp.get_input_details()
    print(f'\n--- 入力 ---')
    for i, d in enumerate(inp_details):
        shape = d['shape']
        dtype = d['dtype']
        print(f'  [{i}] {d["name"]} shape={shape} dtype={dtype}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')

        # 受け入れ条件1: 入力形状チェック
        expected_shape = [1, IMG_SIZE, IMG_SIZE, INPUT_CHANNELS]
        if list(shape) == expected_shape:
            print(f'      [PASS] 入力形状: {list(shape)} == {expected_shape}')
        else:
            print(f'      [FAIL] 入力形状: {list(shape)} != {expected_shape}')
            all_checks_passed = False

        # 受け入れ条件: INT8 型チェック
        if dtype == np.int8:
            print(f'      [PASS] データ型: INT8')
        else:
            print(f'      [FAIL] データ型: {dtype} (INT8が期待されます)')
            all_checks_passed = False

        # 入力バッファサイズ
        buf_size = 1
        for s in shape:
            buf_size *= s
        expected_buf = IMG_SIZE * IMG_SIZE * INPUT_CHANNELS  # 36,864
        print(f'      入力バッファサイズ: {buf_size:,} バイト (期待: {expected_buf:,})')
        if buf_size == expected_buf:
            print(f'      [PASS] 入力バッファサイズ一致')
        else:
            print(f'      [FAIL] 入力バッファサイズ不一致')
            all_checks_passed = False

    # 出力チェック
    out_details = interp.get_output_details()
    print(f'\n--- 出力 ---')
    for i, d in enumerate(out_details):
        print(f'  [{i}] {d["name"]} shape={d["shape"]} dtype={d["dtype"]}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')

    # RGB版との比較
    int8_kb = os.path.getsize(INT8_PATH) / 1024
    rgb_int8_kb = 3149  # RGB版の参考値
    print(f'\n--- RGB版 (3ch) との比較 ---')
    print(f'RGB版 INT8:  {rgb_int8_kb} KB, 入力バッファ: {192*192*3:,} バイト')
    print(f'Gray版 INT8: {int8_kb:.0f} KB, 入力バッファ: {192*192*1:,} バイト')
    print(f'入力バッファ削減: {(1 - 192*192*1 / (192*192*3))*100:.0f}%')

    # 推論テスト
    print(f'\n--- 推論テスト (ダミー入力) ---')
    try:
        inp_idx = inp_details[0]['index']
        dummy = np.zeros(inp_details[0]['shape'], dtype=np.int8)
        interp.set_tensor(inp_idx, dummy)
        interp.invoke()
        for i, d in enumerate(out_details):
            out = interp.get_tensor(d['index'])
            print(f'  出力[{i}]: shape={out.shape}, min={out.min()}, max={out.max()}')
        print(f'  [PASS] 推論正常完了')
    except Exception as e:
        print(f'  [FAIL] 推論エラー: {e}')
        all_checks_passed = False

    # 総合判定
    print(f'\n===============================')
    if all_checks_passed:
        print(f'全チェック PASS: Grayscale (1ch) 入力対応完了')
    else:
        print(f'一部チェック FAIL: 上記のエラーを確認してください')
    print(f'===============================')
else:
    print('INT8モデルが見つかりません。Step 7 を先に実行してください。')

In [ ]:
# MCUコード生成用の情報出力
# MainLoop_obj.cc での tensor 設定に使用する値

if os.path.exists(INT8_PATH):
    import tensorflow as tf
    import numpy as np

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    print('=== MCUコード生成用情報 ===')
    print()
    print('// ai_application_config.h 設定値')

    inp = interp.get_input_details()[0]
    n, h, w, c = inp['shape']
    print(f'#define INPUT_HEIGHT   {h}')
    print(f'#define INPUT_WIDTH    {w}')
    print(f'#define INPUT_CHANNELS {c}    // Grayscale')
    print(f'#define INPUT_SIZE     ({h} * {w} * {c})  // = {h*w*c}')

    qp = inp.get('quantization_parameters', {})
    sc = qp.get('scales', np.array([0.0]))
    zp = qp.get('zero_points', np.array([0]))
    print(f'// Input quantization: scale={float(sc[0]):.8f}, zero_point={int(zp[0])}')

    print()
    print('// MainLoop_obj.cc 出力テンソル設定')
    for i, o in enumerate(interp.get_output_details()):
        qp = o.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([0.0]))
        zp = qp.get('zero_points', np.array([0]))
        print(f'// Output[{i}]: shape={list(o["shape"])}, scale={float(sc[0]):.8f}, zero_point={int(zp[0])}')

---
## Step 9: 成果物ダウンロード

学習済みモデルを Google Drive に保存する。

In [ ]:
# Google Drive に成果物をコピー
import shutil

OUTPUT_DIR = '/content/drive/MyDrive/fall_detection_model_grayscale'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# コピー対象ファイル
files_to_copy = {
    best_pt: 'best_grayscale.pt',
    os.path.join(WORK_DIR, 'train', 'weights', 'last.pt'): 'last_grayscale.pt',
    os.path.join(WORK_DIR, 'train', 'results.png'): 'results.png',
    os.path.join(WORK_DIR, 'train', 'results.csv'): 'results.csv',
    ONNX_PATH: 'model_grayscale.onnx',
    FP32_PATH: 'model_grayscale_fp32.tflite',
    INT8_PATH: 'model_grayscale_int8.tflite',
    model_yaml_path: 'yolov8n-grayscale-fall.yaml',
}

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  {dst_name}: {size_kb:.1f} KB')

print(f'\n=== Google Drive に保存完了 ===')
print(f'場所: {OUTPUT_DIR}')

In [ ]:
# INT8 TFLite モデルを直接ダウンロード
from google.colab import files

if os.path.exists(INT8_PATH):
    files.download(INT8_PATH)
    print('INT8モデルのダウンロードを開始しました')
else:
    print('INT8モデルが見つかりません。Step 7 を先に実行してください。')

---
## まとめ

### 生成される成果物

| ファイル | 説明 |
|---|---|
| `best_grayscale.pt` | 学習済み PyTorch モデル (Grayscale, 最良 mAP) |
| `model_grayscale.onnx` | ONNX 形式モデル (1ch入力) |
| `model_grayscale_fp32.tflite` | TFLite FP32 モデル (1ch入力) |
| `model_grayscale_int8.tflite` | TFLite INT8 量子化モデル (1ch入力) |
| `yolov8n-grayscale-fall.yaml` | カスタムモデル設定 |
| `results.png` | 学習曲線チャート |
| `results.csv` | 学習ログ (CSV) |

### 受け入れ条件の確認

| 条件 | 確認方法 |
|------|----------|
| 入力が [1, 192, 192, 1] | Step 8 の入力形状チェックで確認 |
| Grayscale入力でのmAP@0.5がRGB版と同等以上 | Step 5 のRGB版との比較で確認 |
| INT8量子化後も正常に動作 | Step 8 の推論テストで確認 |

### RGB版との主な違い

| 項目 | RGB版 | Grayscale版 |
|------|-------|-------------|
| 入力形状 | [1, 192, 192, 3] | [1, 192, 192, 1] |
| 入力バッファ | 110,592バイト | 36,864バイト |
| バッファ削減 | - | 66.7% 削減 |
| 色情報 | あり | なし |
| MCUカメラとの整合性 | チャネル複製が必要 | そのまま使用可能 |

### 次のステップ

1. F-003-3b (モデル小型化) と組み合わせて、Grayscale + pico/nano-slim で最終モデルを決定
2. RUHMI/MERA SDK で Ethos-U55 向けに変換
3. 生成コードを `e2studio_CPU0/src/ai_application/` に配置
4. 実機 (EK-RA8P1) での動作確認

### 備考

- F-003-3b の pico/nano-slim ノートブック (`train_yolov8_pico_colab.ipynb`) は
  既に `ch: 1` (Grayscale) 対応済みです
- 本ノートブックは YOLOv8n サイズ (width=0.25) での Grayscale 学習です
- YOLOv8n サイズのINT8モデルは約3MB程度になるため、Arena制約 (432KB) を
  満たすにはF-003-3bの小型化が必須です